# LeetCode 2728 — Count Houses in a Circular Street bằng Python

Link đề: https://leetcode.com/problems/count-houses-in-a-circular-street/ · Độ khó: **Easy** · Chủ đề: Interactive, Simulation

> **Ghi chú Premium:** đây là bài **Premium** (cần tài khoản LeetCode Premium để xem đề gốc và nộp bài). Đề trong notebook này được **viết lại theo hiểu biết** về bài; nếu bạn có tài khoản Premium, hãy **đối chiếu lại** đề, ví dụ và danh sách hàm của lớp `Street` (đặc biệt có `moveLeft()` hay không) với đề gốc. Vì bài dạng **interactive**, notebook tự dựng lớp mô phỏng `Street` để bạn chạy thử và test ngay trên máy.

Skeleton LeetCode (lớp `Street` do hệ thống cung cấp, ta chỉ viết `Solution`):

```python
# class Street:
#     def openDoor(self): ...
#     def closeDoor(self): ...
#     def isDoorOpen(self) -> bool: ...
#     def moveRight(self): ...
#     def moveLeft(self): ...   # (nếu đề gốc có) ta không cần dùng

class Solution:
    def houseCount(self, street: Optional['Street'], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **interactive (tương tác)**: ta **không được xem** dữ liệu `n` hay mảng cửa, chỉ được "điều khiển" qua các hàm `openDoor`, `closeDoor`, `isDoorOpen`, `moveRight`.
2. Hiểu ràng buộc quan trọng của đề: `k` là **cận trên** của số nhà (`n ≤ k`), nên đi `k` bước là chắc chắn thăm **mọi** nhà.
3. Học kỹ thuật **"đánh dấu rồi đi tìm dấu"** (marker): làm sạch trạng thái, đặt dấu duy nhất, rồi đếm số bước để quay lại dấu.
4. Tự **dựng lớp mô phỏng `Street`** (bằng class Python, chỉ số vòng bằng `%`) để chạy thử và đếm số lần gọi hàm.
5. Viết 3 cách: thử từng độ dài `m` (brute force, `O(k²)`), "đóng hết rồi đặt dấu mở" (`O(k)`), và đối xứng "mở hết rồi đặt dấu đóng".
6. Kiểm chứng bằng test ngẫu nhiên có seed (trạng thái cửa, vị trí xuất phát, `n`, `k` ngẫu nhiên hợp lệ).
7. Tránh các lỗi quen thuộc: giả sử mọi cửa ban đầu đóng, dọn sai số bước, kiểm tra trước khi dịch, quên đặt dấu.

## 2. Đề bài & ví dụ

Có một **con đường hình tròn** gồm `n` căn nhà (`n` **chưa biết**). Cửa mỗi nhà lúc đầu có thể **mở** hoặc **đóng** (tuỳ ý). Ban đầu bạn đứng trước cửa một căn nhà nào đó.

Cho đối tượng `street` thuộc lớp `Street` và số nguyên dương `k` là **cận trên** số nhà (số nhà `≤ k`). Lớp `Street` cung cấp:

- `openDoor()`: mở cửa nhà bạn đang đứng trước.
- `closeDoor()`: đóng cửa nhà đó.
- `isDoorOpen()`: trả `True` nếu cửa nhà đó đang mở, ngược lại `False`.
- `moveRight()`: đi sang nhà bên phải (đường tròn, nên sau nhà cuối là nhà đầu).
- `moveLeft()`: đi sang nhà bên trái (nếu đề gốc có; lời giải của ta **không cần**).

Hãy trả về **số nhà `n`** trên đường.

| Ví dụ | street (trạng thái cửa ban đầu, 1 = mở) | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[0, 0, 0, 0]` | 10 | `4` | 4 căn nhà, mọi cửa đóng |
| 2 | `[1, 0, 1, 1, 0]` | 5 | `5` | 5 căn nhà, một số cửa mở |

(Mảng trạng thái chỉ là **dữ liệu để dựng bài test**; lời giải của bạn **không được đọc** mảng đó. *Đối chiếu lại ví dụ với đề gốc nếu có tài khoản Premium.*)

Constraints: `1 <= n <= k <= 10^3`, trong đó `n` là số nhà.

### 2.1. Dựng lớp mô phỏng `Street` để chạy thử

Trên LeetCode, lớp `Street` có sẵn. Để tự chạy, ta dựng **lớp mô phỏng**:

- `doors`: danh sách trạng thái cửa (`1` mở, `0` đóng) — **giấu** bên trong (tiền tố `_`).
- `start`: nhà ban đầu mình đứng trước.
- `calls`: bộ đếm số lần gọi từng hàm (để so sánh chi phí các cách).
- `move_limit`: (tuỳ chọn) nếu gọi `moveRight/moveLeft` quá số lần này thì báo lỗi — dùng để phát hiện vòng lặp vô hạn khi test.
- Các hàm bắt đầu bằng `true_count`, `snapshot` chỉ để **kiểm tra / in ra** trong notebook; lời giải **không được gọi**.

In [1]:
from typing import Optional


class Street:
    """Mô phỏng đường tròn n căn nhà. Lời giải chỉ được dùng 4 hàm: openDoor, closeDoor, isDoorOpen, moveRight."""

    def __init__(self, doors, start=0, move_limit=None):
        assert len(doors) >= 1 and all(d in (0, 1) for d in doors)
        self._doors = list(doors)           # 1 = mở, 0 = đóng (giấu bên trong)
        self._n = len(doors)
        self._pos = start % self._n
        self._move_limit = move_limit
        self.calls = {"openDoor": 0, "closeDoor": 0, "isDoorOpen": 0, "moveRight": 0, "moveLeft": 0}

    # ----- các hàm đề cho phép -----
    def openDoor(self):
        self.calls["openDoor"] += 1
        self._doors[self._pos] = 1

    def closeDoor(self):
        self.calls["closeDoor"] += 1
        self._doors[self._pos] = 0

    def isDoorOpen(self) -> bool:
        self.calls["isDoorOpen"] += 1
        return self._doors[self._pos] == 1

    def moveRight(self):
        self.calls["moveRight"] += 1
        self._check_limit()
        self._pos = (self._pos + 1) % self._n

    def moveLeft(self):
        self.calls["moveLeft"] += 1
        self._check_limit()
        self._pos = (self._pos - 1) % self._n

    # ----- hàm hỗ trợ kiểm tra (lời giải KHÔNG được gọi) -----
    def _check_limit(self):
        if self._move_limit is not None and self.calls["moveRight"] + self.calls["moveLeft"] > self._move_limit:
            raise RuntimeError("vượt giới hạn số lần di chuyển (nghi ngờ vòng lặp vô hạn)")

    def true_count(self) -> int:
        return self._n

    def snapshot(self) -> str:
        cells = []
        for i, d in enumerate(self._doors):
            cells.append(f"[{d}]" if i == self._pos else f" {d} ")
        return "".join(cells)

    def moves(self) -> int:
        return self.calls["moveRight"] + self.calls["moveLeft"]


st = Street([1, 0, 1, 1, 0], start=2)
print(st.snapshot(), "| đang ở nhà", st._pos)       # [..] đánh dấu nhà đang đứng
print(st.isDoorOpen())                               # True
st.closeDoor(); print(st.isDoorOpen())               # False
st.moveRight(); print(st.snapshot())

 1  0 [1] 1  0  | đang ở nhà 2
True
False
 1  0  0 [1] 0 


## 3. Kiến thức nền

### 3.1. Bài "interactive": chỉ được hỏi qua hàm cho sẵn

Ở bài thường, đề cho thẳng dữ liệu (`nums`, `s`...). Ở bài **interactive**, đề cho **một đối tượng** và ta chỉ biết được thông tin bằng cách **gọi hàm** của nó, đôi khi hàm còn **thay đổi trạng thái** (ở đây `openDoor/closeDoor` thay đổi cửa, `moveRight` đổi vị trí). Quan trọng: ta **không biết `n`**, chỉ biết `n ≤ k`.

Tư duy: coi đối tượng như "hộp đen"; ta cần một **chiến thuật** gọi hàm đúng thứ tự để suy ra điều cần tìm.

In [2]:
street = Street([0, 1, 1], start=0)
# Từ góc nhìn của lời giải, ta chỉ làm được những việc sau:
print(street.isDoorOpen())     # nhà 0: False
street.openDoor()
print(street.isDoorOpen())     # True (vừa mở)
street.moveRight()
print(street.isDoorOpen())     # nhà 1: True (ban đầu đã mở)
print(street.calls)

False
True
True
{'openDoor': 1, 'closeDoor': 0, 'isDoorOpen': 3, 'moveRight': 1, 'moveLeft': 0}


### 3.2. Lớp (class) và đối tượng (object) trong Python

`class Street:` định nghĩa một "khuôn". `Street(...)` tạo một **đối tượng** (instance). Các hàm bên trong lớp gọi là **phương thức (method)**, tham số đầu `self` là chính đối tượng. Các biến lưu bằng `self.x` là **thuộc tính** (trạng thái bên trong, như `_pos`, `_doors`). Tiền tố `_` là quy ước "nội bộ, đừng đụng".

In [3]:
class Counter3:
    def __init__(self):
        self.value = 0          # thuộc tính (trạng thái bên trong)
    def inc(self):              # phương thức
        self.value += 1

c = Counter3()
c.inc(); c.inc()
print(c.value)                  # 2

2


### 3.3. Đường tròn bằng phép chia lấy dư `%`

Với `n` nhà đánh số `0..n-1`, nhà bên phải của `p` là `(p + 1) % n`; bên trái là `(p - 1) % n` (Python cho kết quả không âm). Nhờ vậy sau nhà cuối `n - 1` ta quay về `0`.

In [4]:
n = 5
print([(p + 1) % n for p in range(n)])        # [1, 2, 3, 4, 0]
print([(p - 1) % n for p in range(n)])        # [4, 0, 1, 2, 3]
p = 3
for _ in range(7):
    p = (p + 1) % n
print(p)                                       # (3 + 7) % 5 = 0

[1, 2, 3, 4, 0]
[4, 0, 1, 2, 3]
0


### 3.4. `for _ in range(k)` và thoát sớm bằng `return`

`for _ in range(k):` lặp `k` lần mà không cần dùng biến đếm (quy ước dùng `_`). Muốn dừng sớm khi tìm thấy: dùng `return` ngay trong vòng lặp. Có biến đếm bước thì viết `for step in range(1, k + 1):` để `step` chạy `1, 2, ..., k`.

In [5]:
for _ in range(3):
    print("lặp", end=" ")
print()

def find_first_even(k):
    for step in range(1, k + 1):
        if step % 2 == 0:
            return step           # thoát luôn
    return -1

print(find_first_even(5), find_first_even(1))     # 2 -1

lặp lặp lặp 
2 -1


### 3.5. Ý tưởng "marker" (dấu) — quen từ danh sách liên kết vòng

Muốn biết đi bao nhiêu bước thì **quay về điểm xuất phát**, ta cần **nhận ra điểm xuất phát**. Cách làm: tạo ra một **dấu duy nhất** có thể nhìn thấy, rồi đi cho tới khi gặp lại dấu. Dấu ở đây là **trạng thái cửa**: nếu mọi cửa khác **đóng** mà đúng một cửa **mở** thì cửa mở đó là dấu. Cùng ý tưởng với việc phát hiện chu trình / đếm nút trong danh sách liên kết vòng.

In [6]:
# Hình dung trên một vòng 5 ô: 1 = dấu duy nhất, 0 = còn lại
ring = [0, 0, 1, 0, 0]
pos = 2                      # đang đứng ở dấu
steps = 0
while True:
    pos = (pos + 1) % len(ring)
    steps += 1
    if ring[pos] == 1:       # gặp lại dấu
        break
print("số ô của vòng =", steps)       # 5

số ô của vòng = 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Nếu mọi cửa ban đầu **đóng** và mình **mở** cửa nhà xuất phát (làm dấu), ta chỉ cần đi sang phải đến khi thấy cửa mở: số bước chính là `n`. Quá dễ!

Vấn đề: đề cho phép **một số cửa ban đầu đã mở** — lúc đó "gặp cửa mở" có thể không phải dấu của ta. Cách khắc phục: **đóng tất cả cửa trước** (làm sạch), rồi mới đặt dấu.

Nhưng ta không biết `n`, làm sao đóng **tất cả**? Dùng giả thiết: `n ≤ k`. Đi `k` bước và đóng mọi cửa trên đường ⇒ vì `k ≥ n`, ta đi vòng quanh **ít nhất một vòng** nên **đã đóng mọi cửa**. Rồi mở cửa nhà hiện tại làm dấu, đi sang phải đếm tới khi cửa mở lại.

Tính tay trên `[1, 0, 1]` (`n = 3`), `k = 3`, đứng ở nhà 0:

| giai đoạn | việc làm | trạng thái cửa |
|---|---|---|
| dọn | `đóng, sang phải` ×3 | `[0,0,0]` (về lại nhà 0) |
| dấu | mở nhà 0 | `[1,0,0]` |
| đếm | sang phải: nhà 1 đóng, nhà 2 đóng, nhà 0 **mở** ⇒ 3 bước | đáp án 3 |

In [7]:
street = Street([1, 0, 1], start=0)
k = 3
for _ in range(k):
    street.closeDoor()
    street.moveRight()
print("sau khi dọn:", street.snapshot())
street.openDoor()
print("đặt dấu    :", street.snapshot())
for step in range(1, k + 1):
    street.moveRight()
    print(f"bước {step}:", street.snapshot(), "mở?", street.isDoorOpen())
    if street.isDoorOpen():
        print("n =", step)
        break

sau khi dọn: [0] 0  0 
đặt dấu    : [1] 0  0 
bước 1:  1 [0] 0  mở? False
bước 2:  1  0 [0] mở? False
bước 3: [1] 0  0  mở? True
n = 3


## 5. Cách 1 — Brute Force / baseline: thử từng độ dài `m`

**Ý tưởng:** ta thử **từng giá trị có thể** của `n` là `m = 1, 2, ..., k`. Với mỗi `m`: (1) **làm sạch**: đóng `k` cửa liên tiếp ⇒ mọi cửa đóng; (2) **đặt dấu**: mở cửa nhà hiện tại; (3) đi sang phải đúng `m` bước rồi hỏi `isDoorOpen()`. Nếu **mở**, ta đã quay về dấu sau đúng `m` bước nên `m` là độ dài vòng ⇒ trả `m`.

Tại sao gặp dấu chỉ khi `m = n`? Vì mọi cửa khác đóng, sau `m` bước ta ở nhà `(h + m) mod n` và nó chính là nhà dấu `h` khi và chỉ khi `n` chia hết `m`; `m` nhỏ nhất thoả là `m = n`.

In [8]:
def house_count_brute(street, k: int) -> int:
    for m in range(1, k + 1):
        for _ in range(k):               # làm sạch lại từ đầu cho mỗi m
            street.closeDoor()
            street.moveRight()
        street.openDoor()                # đặt dấu
        for _ in range(m):               # đi đúng m bước
            street.moveRight()
        if street.isDoorOpen():          # đã quay về dấu?
            return m
    return -1

for doors, k in [([0, 0, 0, 0], 10), ([1, 0, 1, 1, 0], 5)]:
    st = Street(doors, start=1)
    print(doors, "k =", k, "->", house_count_brute(st, k), "| số lần moveRight:", st.calls["moveRight"])

[0, 0, 0, 0] k = 10 -> 4 | số lần moveRight: 50
[1, 0, 1, 1, 0] k = 5 -> 5 | số lần moveRight: 40


**Độ phức tạp:** với mỗi `m` ta tốn `k` (dọn) `+ m` (đi) lần `moveRight`; có `n` giá trị `m` được thử ⇒ tổng `n·k + n(n+1)/2` lần di chuyển ⇒ Time `O(n · k)`, xấu nhất `O(k²)` (`k ≤ 1000` ⇒ cỡ `10^6` thao tác). Space `O(1)`.

**Nút thắt:** mỗi lần thử `m` ta **dọn lại** `k` cửa dù **mọi cửa đã đóng sẵn** từ lần trước (chỉ có dấu cần đóng lại). Chỉ cần dọn **một lần**, rồi đi cho tới khi gặp dấu — không cần đoán `m`.

## 6. Ý tưởng thuật toán / bước đột phá

Vì dọn xong thì **chỉ có đúng một cửa mở (dấu)**, câu hỏi "đi sang phải bao nhiêu bước thì gặp cửa mở?" tự trả lời `n`. Không cần thử từng `m`: ta cứ **đi và hỏi `isDoorOpen()` sau mỗi bước**, dừng ở lần đầu thấy mở.

Quy trình hai pha:

1. **Pha dọn (`k` bước):** `closeDoor(); moveRight()` lặp `k` lần ⇒ đóng hết mọi cửa (vì `k ≥ n`).
2. **Đặt dấu:** `openDoor()` tại nhà hiện tại.
3. **Pha đếm:** lặp tối đa `k` lần: `moveRight()`; nếu `isDoorOpen()` ⇒ trả số bước đã đi.

Tổng số lần di chuyển: `k + n ≤ 2k`.

In [9]:
street = Street([0, 1, 1], start=2)        # n = 3, k = 5 (k > n cũng được)
k = 5
for _ in range(k):
    street.closeDoor(); street.moveRight()
street.openDoor()
count = 0
for step in range(1, k + 1):
    street.moveRight()
    if street.isDoorOpen():
        count = step
        break
print("n =", count, "| tổng moveRight:", street.calls["moveRight"])      # 3 | 8

n = 3 | tổng moveRight: 8


## 7. Các cách tiếp theo

Ta có Cách 1 (thử từng `m`) và sẽ xem: (2) **đóng hết rồi đặt dấu mở** (cách chuẩn `O(k)`), (3) **mở hết rồi đặt dấu đóng** (đối xứng).

### Cách 2 — Đóng hết, đặt dấu mở, đi cho tới khi gặp cửa mở

Giải thích từng đoạn:

- `for _ in range(k): street.closeDoor(); street.moveRight()` — đi `k` nhà, đóng cửa từng nhà. Vì `k ≥ n` nên đi hết (và có thể lặp lại) mọi nhà ⇒ **mọi cửa đóng**.
- `street.openDoor()` — ta đang đứng trước một nhà (nhà nào cũng được); mở cửa nhà này làm **dấu duy nhất**.
- `for step in range(1, k + 1): street.moveRight(); if street.isDoorOpen(): return step` — mỗi bước sang phải rồi hỏi; vì chỉ dấu mở, lần đầu thấy mở là **quay về dấu** sau đúng `n` bước.
- `return -1` không bao giờ chạy (vì `n ≤ k`), chỉ để vòng lặp luôn có `return`.

In [10]:
def house_count_close_mark(street, k: int) -> int:
    for _ in range(k):
        street.closeDoor()
        street.moveRight()
    street.openDoor()                       # dấu
    for step in range(1, k + 1):
        street.moveRight()                  # dịch trước...
        if street.isDoorOpen():             # ...rồi mới hỏi
            return step
    return -1

for doors, k in [([0, 0, 0, 0], 10), ([1, 0, 1, 1, 0], 5)]:
    st = Street(doors, start=3)
    print(doors, "k =", k, "->", house_count_close_mark(st, k), "| số lần moveRight:", st.calls["moveRight"])

[0, 0, 0, 0] k = 10 -> 4 | số lần moveRight: 14
[1, 0, 1, 1, 0] k = 5 -> 5 | số lần moveRight: 10


In [11]:
from typing import Optional

class SolutionCloseMark:
    def houseCount(self, street: Optional['Street'], k: int) -> int:
        for _ in range(k):
            street.closeDoor()
            street.moveRight()
        street.openDoor()
        for step in range(1, k + 1):
            street.moveRight()
            if street.isDoorOpen():
                return step
        return -1

print(SolutionCloseMark().houseCount(Street([1, 0, 1, 1, 0], start=4), 5))

5


**Độ phức tạp:** pha dọn `k` lần di chuyển, pha đếm `n ≤ k` lần ⇒ Time `O(k)` (tổng tối đa `2k` thao tác di chuyển), Space `O(1)` (chỉ vài biến). Đây là cách tối ưu theo cấp độ lớn: ta **buộc** phải đi ít nhất `k` bước vì không biết `n`.

### Cách 3 — Đối xứng: mở hết, đặt dấu **đóng**, đi cho tới khi gặp cửa **đóng**

Cùng ý với Cách 2 nhưng đảo vai trò: pha dọn **mở** mọi cửa (`openDoor()`), rồi `closeDoor()` ở nhà hiện tại làm dấu (duy nhất đóng), rồi đi sang phải cho tới khi `isDoorOpen()` trả `False`. Dùng cách này để nhớ rằng "dấu" có thể là cửa mở hoặc đóng, miễn là **khác mọi cửa còn lại**.

In [12]:
def house_count_open_mark(street, k: int) -> int:
    for _ in range(k):
        street.openDoor()
        street.moveRight()
    street.closeDoor()                      # dấu = cửa đóng duy nhất
    for step in range(1, k + 1):
        street.moveRight()
        if not street.isDoorOpen():
            return step
    return -1

for doors, k in [([0, 0, 0, 0], 10), ([1, 0, 1, 1, 0], 5)]:
    st = Street(doors, start=0)
    print(doors, "k =", k, "->", house_count_open_mark(st, k), "| số lần moveRight:", st.calls["moveRight"])

[0, 0, 0, 0] k = 10 -> 4 | số lần moveRight: 14
[1, 0, 1, 1, 0] k = 5 -> 5 | số lần moveRight: 10


In [13]:
from typing import Optional

class SolutionOpenMark:
    def houseCount(self, street: Optional['Street'], k: int) -> int:
        for _ in range(k):
            street.openDoor()
            street.moveRight()
        street.closeDoor()
        for step in range(1, k + 1):
            street.moveRight()
            if not street.isDoorOpen():
                return step
        return -1

print(SolutionOpenMark().houseCount(Street([0, 0, 0, 0], start=2), 10))

4


**Độ phức tạp:** y hệt Cách 2: Time `O(k)`, Space `O(1)`.

### So sánh chi phí đo được (số lần gọi hàm)

Trong lớp mô phỏng `Street` có bộ đếm `calls`, nên ta **đo** số lần di chuyển của từng cách trên cùng một bài: `n = 50` nhà, `k = 1000`.

In [14]:
import random

random.seed(2728)
doors50 = [random.randint(0, 1) for _ in range(50)]
for name, f in [("Cách 1 (thử từng m)", house_count_brute),
                ("Cách 2 (đóng hết, dấu mở)", house_count_close_mark),
                ("Cách 3 (mở hết, dấu đóng)", house_count_open_mark)]:
    st = Street(doors50, start=7)
    ans = f(st, 1000)
    print(f"{name:<28} đáp án={ans}  moveRight={st.calls['moveRight']}")

Cách 1 (thử từng m)          đáp án=50  moveRight=51275
Cách 2 (đóng hết, dấu mở)    đáp án=50  moveRight=1050
Cách 3 (mở hết, dấu đóng)    đáp án=50  moveRight=1050


## 8. TRACE từng bước

Trace Cách 2 với `street = [1, 0, 1, 1, 0]` (`n = 5`), `k = 5`, đứng ở nhà `0`. Dấu `[ ]` là nhà đang đứng.

| giai đoạn | bước | hành động | trạng thái |
|---|---|---|---|
| đầu | | | `[1] 0  1  1  0` |
| dọn | 1 | đóng nhà 0, sang phải | ` 0 [0] 1  1  0` |
| dọn | 2 | đóng nhà 1, sang phải | ` 0  0 [1] 1  0` |
| dọn | 3 | đóng nhà 2, sang phải | ` 0  0  0 [1] 0` |
| dọn | 4 | đóng nhà 3, sang phải | ` 0  0  0  0 [0]` |
| dọn | 5 | đóng nhà 4, sang phải (về nhà 0) | `[0] 0  0  0  0` |
| dấu | — | mở nhà 0 | `[1] 0  0  0  0` |
| đếm | 1 | sang phải: nhà 1 đóng | |
| đếm | 2, 3, 4 | nhà 2, 3, 4 đóng | |
| đếm | 5 | sang phải về nhà 0: **mở** | trả `5` |

In [15]:
def trace_close_mark(doors, k, start=0):
    st = Street(doors, start=start)
    print("đầu          :", st.snapshot())
    for i in range(1, k + 1):
        st.closeDoor()
        st.moveRight()
        print(f"dọn bước {i}   :", st.snapshot())
    st.openDoor()
    print("đặt dấu mở   :", st.snapshot())
    for step in range(1, k + 1):
        st.moveRight()
        opened = st.isDoorOpen()
        print(f"đếm bước {step}   :", st.snapshot(), "mở?", opened)
        if opened:
            print("=> đáp án", step, "| thực tế n =", st.true_count())
            return step
    return -1

trace_close_mark([1, 0, 1, 1, 0], 5, start=0)

đầu          : [1] 0  1  1  0 
dọn bước 1   :  0 [0] 1  1  0 
dọn bước 2   :  0  0 [1] 1  0 
dọn bước 3   :  0  0  0 [1] 0 
dọn bước 4   :  0  0  0  0 [0]
dọn bước 5   : [0] 0  0  0  0 
đặt dấu mở   : [1] 0  0  0  0 
đếm bước 1   :  1 [0] 0  0  0  mở? False
đếm bước 2   :  1  0 [0] 0  0  mở? False
đếm bước 3   :  1  0  0 [0] 0  mở? False
đếm bước 4   :  1  0  0  0 [0] mở? False
đếm bước 5   : [1] 0  0  0  0  mở? True
=> đáp án 5 | thực tế n = 5


5

In [16]:
# Trace thứ hai: k > n và xuất phát giữa chừng (n = 3, k = 5)
trace_close_mark([0, 1, 1], 5, start=2)

đầu          :  0  1 [1]
dọn bước 1   : [0] 1  0 
dọn bước 2   :  0 [1] 0 
dọn bước 3   :  0  0 [0]
dọn bước 4   : [0] 0  0 
dọn bước 5   :  0 [0] 0 
đặt dấu mở   :  0 [1] 0 
đếm bước 1   :  0  1 [0] mở? False
đếm bước 2   : [0] 1  0  mở? False
đếm bước 3   :  0 [1] 0  mở? True
=> đáp án 3 | thực tế n = 3


3

## 9. Code nộp LeetCode

Dùng Cách 2 (đóng hết, dấu mở). Trên LeetCode lớp `Street` có sẵn, nên chỉ cần `class Solution`. Ở notebook ta đã dựng `Street` ở trên nên gọi thử được.

In [17]:
from typing import Optional

class Solution:
    def houseCount(self, street: Optional['Street'], k: int) -> int:
        for _ in range(k):
            street.closeDoor()
            street.moveRight()
        street.openDoor()
        for step in range(1, k + 1):
            street.moveRight()
            if street.isDoorOpen():
                return step
        return -1

In [18]:
print(Solution().houseCount(Street([0, 0, 0, 0], start=0), 10))     # 4
print(Solution().houseCount(Street([1, 0, 1, 1, 0], start=0), 5))   # 5

4
5


## 10. Vì sao thuật toán đúng

Gọi `n` là số nhà (chưa biết), `n ≤ k`, và `h` là nhà ta đứng lúc bắt đầu pha dọn.

**Pha dọn đóng mọi cửa.** Vòng `for _ in range(k)` thăm các nhà `h, h+1, ..., h+k-1` (chỉ số theo modulo `n`), tức `k` nhà **liên tiếp**. Vì `k ≥ n`, `k` nhà liên tiếp trên một đường tròn `n` nhà **bao phủ mọi nhà ít nhất một lần**. Mỗi lần thăm ta `closeDoor()` ⇒ khi xong, **mọi cửa đóng** (kể cả cửa ban đầu mở, vì đã được đóng khi thăm).

**Sau pha dọn.** Ta đang đứng ở nhà `M = (h + k) mod n`. `openDoor()` ⇒ trạng thái: **đúng một cửa mở** (nhà `M`).

**Pha đếm.** Sau `t` bước sang phải ta ở nhà `(M + t) mod n`. Cửa ở đó mở **khi và chỉ khi** đó là nhà `M`, tức `(M + t) mod n = M`, tức `n` chia hết `t`. `t` dương nhỏ nhất như vậy là `t = n`. Vì `n ≤ k`, vòng `for step in range(1, k + 1)` **chắc chắn** chạm `step = n` và trả về `n`. Các bước `t < n` đều gặp cửa đóng nên không trả sớm.

**Hiểu vì sao cần `n ≤ k`.** Nếu `k < n` thì pha dọn có thể bỏ sót cửa mở (cửa nằm ngoài `k` nhà đã dọn) và dấu bị "nhiễu" — thuật toán sai. Đề đảm bảo `n ≤ k` chính để thuật toán này đúng.

## 11. Phân tích Time / Space Complexity

Đo theo **số lần gọi hàm của `street`** (mỗi lần gọi coi là `O(1)`):

| Cách | Số lần di chuyển | Time | Space |
|---|---|---|---|
| 1. thử từng `m` | `n·k + n(n+1)/2` | `O(n·k)` (xấu nhất `O(k²)`) | `O(1)` |
| 2. đóng hết, dấu mở | `k + n ≤ 2k` | `O(k)` | `O(1)` |
| 3. mở hết, dấu đóng | `k + n ≤ 2k` | `O(k)` | `O(1)` |

Cận dưới: vì không biết `n`, để chắc chắn đã chạm mọi nhà ta phải đi ít nhất khoảng `k` bước nên `O(k)` là tối ưu cho hướng "làm sạch toàn bộ". Space `O(1)` vì chỉ dùng vài biến đếm, không lưu danh sách nhà.

## 12. Bộ test

### 12.1. Ví dụ của đề

In [19]:
sols = [SolutionCloseMark(), SolutionOpenMark(), Solution()]
funcs = [house_count_brute, house_count_close_mark, house_count_open_mark] + [s.houseCount for s in sols]

for f in funcs:
    assert f(Street([0, 0, 0, 0]), 10) == 4
    assert f(Street([1, 0, 1, 1, 0]), 5) == 5
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case (chỉ dùng input hợp lệ: `1 <= n <= k <= 1000`)

- Một nhà: `[0]`, `[1]` với `k = 1` và `k` lớn (ví dụ 1000). Một nhà là nhà "bên phải của chính nó".
- `k = n` (cận trên chặt): `[1, 1, 1]`, `k = 3`.
- Toàn cửa mở / toàn cửa đóng.
- Xuất phát ở nhà bất kỳ (đầu, giữa, cuối).
- `n = 1000`, `k = 1000` (kích thước lớn nhất).
- `n` nhỏ nhưng `k` rất lớn (`n = 3`, `k = 1000`).

In [20]:
cases = [
    ([0], 1, 0), ([1], 1, 0), ([0], 1000, 0), ([1], 1000, 0),
    ([1, 1, 1], 3, 0), ([1, 1, 1], 3, 2), ([0, 0, 0], 3, 1),
    ([1, 0, 1, 1, 0], 5, 0), ([1, 0, 1, 1, 0], 5, 4), ([1, 0, 1, 1, 0], 7, 2),
    ([0, 1, 1], 1000, 1),
    ([i % 2 for i in range(1000)], 1000, 500),
    ([1] * 1000, 1000, 999),
    ([0] * 1000, 1000, 0),
]
fast = [house_count_close_mark, house_count_open_mark] + [s.houseCount for s in sols]
for doors, k, start in cases:
    n = len(doors)
    assert n <= k <= 1000 and 1 <= n
    for f in fast:
        assert f(Street(doors, start=start), k) == n, (f, n, k, start)
# cách 1 chậm O(n*k): chỉ kiểm các ca nhỏ
for doors, k, start in cases:
    if len(doors) <= 7 and k <= 10:
        assert house_count_brute(Street(doors, start=start), k) == len(doors)
print("Edge case OK:", len(cases), "trường hợp")

Edge case OK: 14 trường hợp


### 12.3. Phản ví dụ cho cách làm sai

Các cách sai hay gặp (chi tiết ở mục 13): (a) **không dọn** mà đặt dấu luôn, (b) dọn **quá ít** bước (`k // 2`), (c) hỏi `isDoorOpen()` **trước** khi dịch. Dùng `Street` mô phỏng để chứng minh chúng sai.

In [21]:
def sai_khong_don(street, k):
    street.openDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():
            return step
    return -1

def sai_don_it(street, k):
    for _ in range(k // 2):                  # chỉ dọn một nửa
        street.closeDoor()
        street.moveRight()
    street.openDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():
            return step
    return -1

def sai_hoi_truoc(street, k):
    for _ in range(k):
        street.closeDoor()
        street.moveRight()
    street.openDoor()
    for step in range(1, k + 1):
        if street.isDoorOpen():              # hỏi TRƯỚC khi dịch -> luôn thấy dấu ngay
            return step
        street.moveRight()
    return -1

print("không dọn :", sai_khong_don(Street([1, 0, 1, 1, 0]), 5), "(đúng 5)")
print("dọn ít    :", sai_don_it(Street([1, 1, 1, 1]), 4), "(đúng 4)")
print("hỏi trước :", sai_hoi_truoc(Street([0, 0, 0, 0]), 4), "(đúng 4)")
assert sai_khong_don(Street([1, 0, 1, 1, 0]), 5) != 5
assert sai_don_it(Street([1, 1, 1, 1]), 4) != 4
assert sai_hoi_truoc(Street([0, 0, 0, 0]), 4) != 4

không dọn : 2 (đúng 5)
dọn ít    : 1 (đúng 4)
hỏi trước : 1 (đúng 4)


### 12.4. Đối chiếu nhiều input ngẫu nhiên (có seed)

Sinh bài test **hợp lệ theo ràng buộc đề**: `n` từ 1 đến 40, `k` từ `n` đến `n + 60` (đôi khi `k = n`, đôi khi `k` rất lớn), trạng thái cửa ngẫu nhiên (xác suất mở khác nhau mỗi test để có cả "gần như toàn mở" lẫn "gần như toàn đóng"), nhà xuất phát ngẫu nhiên. Kiểm tra: đáp án đúng `n`, **không** dùng `moveLeft`, và số lần di chuyển của Cách 2 và 3 **đúng bằng `k + n`**. Có đặt `move_limit` để bắt vòng lặp vô hạn.

In [22]:
random.seed(27280)
cnt = 0
for _ in range(1500):
    n = random.randint(1, 40)
    k = n if random.random() < 0.3 else n + random.randint(0, 60)
    p = random.choice([0.0, 0.2, 0.5, 0.8, 1.0])
    doors = [1 if random.random() < p else 0 for _ in range(n)]
    start = random.randrange(n)
    for f in fast:
        st = Street(doors, start=start, move_limit=2 * k + 5)
        assert f(st, k) == n, (n, k, start, doors, f)
        assert st.calls["moveLeft"] == 0
        assert st.moves() == k + n, (st.moves(), k, n)
    cnt += 1
print("OK", cnt, "test ngẫu nhiên: đáp án đúng, không dùng moveLeft, số di chuyển = k + n")

# Cách 1 (chậm) kiểm riêng trên input nhỏ
for _ in range(200):
    n = random.randint(1, 8)
    k = n + random.randint(0, 4)
    doors = [random.randint(0, 1) for _ in range(n)]
    st = Street(doors, start=random.randrange(n))
    assert house_count_brute(st, k) == n
print("OK 200 test nhỏ cho Cách 1")

OK 1500 test ngẫu nhiên: đáp án đúng, không dùng moveLeft, số di chuyển = k + n
OK 200 test nhỏ cho Cách 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Giả sử mọi cửa ban đầu đều đóng (không dọn)

In [23]:
def sai_khong_don_cua(street, k):
    street.openDoor()                        # đặt dấu
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():              # có thể gặp cửa mở "có sẵn", không phải dấu
            return step
    return -1

st = Street([1, 0, 1, 1, 0], start=0)
print("sai :", sai_khong_don_cua(st, 5))                        # 2
print("đúng:", Solution().houseCount(Street([1, 0, 1, 1, 0], start=0), 5))   # 5
assert sai_khong_don_cua(Street([1, 0, 1, 1, 0], start=0), 5) == 2

sai : 2
đúng: 5


**Vì sao sai:** nhà 2 đã mở sẵn nên ta nhầm nó là dấu và trả `2`. Đề nói rõ cửa ban đầu có thể mở hoặc đóng ⇒ phải **dọn** để chỉ còn đúng một dấu.

### Lỗi 2: Dọn quá ít bước (không dùng đủ `k`)

In [24]:
def sai_don_chua_du(street, k):
    for _ in range(k // 2):                  # SAI: chưa chắc đã đi hết một vòng
        street.closeDoor()
        street.moveRight()
    street.openDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():
            return step
    return -1

print("sai :", sai_don_chua_du(Street([1, 1, 1, 1], start=0), 4))         # 1
print("đúng:", Solution().houseCount(Street([1, 1, 1, 1], start=0), 4))   # 4
assert sai_don_chua_du(Street([1, 1, 1, 1], start=0), 4) != 4

sai : 1
đúng: 4


**Vì sao sai:** ta không biết `n`, chỉ biết `n ≤ k`. Chỉ có `k` bước mới chắc chắn đi hết vòng. Dọn `k // 2` bước có thể bỏ sót nhà, mà nhà còn cửa mở đó sẽ bị nhầm là dấu.

### Lỗi 3: Hỏi `isDoorOpen()` trước khi dịch (lệch một bước)

In [25]:
def sai_hoi_truoc_khi_dich(street, k):
    for _ in range(k):
        street.closeDoor()
        street.moveRight()
    street.openDoor()
    count = 0
    while not street.isDoorOpen():           # SAI: đang đứng đúng nhà dấu nên vòng lặp không chạy
        street.moveRight()
        count += 1
    return count

print("sai :", sai_hoi_truoc_khi_dich(Street([0, 0, 0, 0], start=0), 4))        # 0
print("đúng:", Solution().houseCount(Street([0, 0, 0, 0], start=0), 4))          # 4
assert sai_hoi_truoc_khi_dich(Street([0, 0, 0, 0], start=0), 4) == 0

sai : 0
đúng: 4


**Vì sao sai:** sau khi đặt dấu ta **đang đứng trước nhà dấu**, nên `isDoorOpen()` đúng ngay lập tức và kết quả `0` (hoặc `1` nếu đếm lệch). Phải **dịch sang phải trước, rồi mới hỏi**.

### Lỗi 4: Quên đặt dấu

In [26]:
def sai_quen_dat_dau(street, k):
    for _ in range(k):
        street.closeDoor()
        street.moveRight()
    # SAI: quên street.openDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():
            return step
    return -1

print("sai :", sai_quen_dat_dau(Street([1, 0, 1], start=0), 3))        # -1 (không bao giờ thấy cửa mở)
print("đúng:", Solution().houseCount(Street([1, 0, 1], start=0), 3))   # 3
assert sai_quen_dat_dau(Street([1, 0, 1], start=0), 3) == -1

sai : -1
đúng: 3


**Vì sao sai:** sau khi đóng hết, không còn cửa mở nào để nhận ra. Vòng đếm chạy hết `k` bước rồi trả `-1`. (Nếu dùng `while True` thay vì `for` có giới hạn thì chương trình **treo mãi mãi** — đó là lý do ta nên dùng vòng lặp có cận `k`.)

### Lỗi 5: Trả về `k` thay vì số bước ở pha đếm

In [27]:
def sai_tra_ve_k(street, k):
    for _ in range(k):
        street.closeDoor()
        street.moveRight()
    street.openDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():
            break
    return k                                   # SAI: số nhà là `step`, không phải k

print("sai :", sai_tra_ve_k(Street([0, 0, 0], start=0), 10))     # 10
print("đúng:", Solution().houseCount(Street([0, 0, 0], start=0), 10))   # 3
assert sai_tra_ve_k(Street([0, 0, 0], start=0), 10) == 10

sai : 10
đúng: 3


**Vì sao sai:** `k` chỉ là **cận trên**, không phải đáp án. Số nhà là số bước từ dấu đi tới lúc **quay lại dấu**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài **interactive** + vòng tròn + biết cận trên `k` ⇒ **làm sạch → đặt dấu → đếm bước tới khi gặp dấu**.
- Mọi bài "đếm độ dài chu trình khi chỉ nhìn cục bộ" đều cần một **dấu phân biệt được**. Hỏi: "mình tạo dấu bằng cách nào?".
- Nhớ thứ tự **dịch → rồi hỏi** (không hỏi trước khi dịch).
- Dùng **vòng lặp có cận** (`for ... in range(k)`) thay vì `while True` để tránh treo khi sai logic.
- Tự dựng **lớp mô phỏng** có bộ đếm lời gọi: giúp test, đo chi phí và bắt lỗi.
- Đừng quên **giả thiết** của đề (ở đây `n ≤ k`): nó là lý do thuật toán đúng.

## 15. Biến thể của bài

1. **Không biết `k`** (chỉ biết có tối đa hữu hạn nhà): cần `moveLeft()` để quay lại đánh dấu, tốn `O(n²)` (xem Bài 5(b)).
2. **Đường thẳng thay vì tròn**: đếm nhà đến khi hết đường — cần cách phát hiện "đầu mút".
3. **Danh sách liên kết vòng**: đếm số nút khi chỉ có `next` (kỹ thuật **Floyd hai con trỏ**, LeetCode 141 / 142).
4. **Tìm nhà có cửa mở thứ `j`** hoặc **số cửa đang mở lúc đầu** (đếm trước khi dọn: tăng đếm khi `isDoorOpen()`).
5. **Giảm số lần di chuyển**: nếu có `moveLeft` ta có thể "quay lại" thay vì đi hết vòng.
6. **Nhiều người đứng cùng lúc**: bài phân tán / đồng bộ hoá.

In [28]:
# Thống kê số lần di chuyển theo k (Cách 2 luôn là k + n)
for n, k in [(5, 5), (5, 50), (50, 1000)]:
    st = Street([1] * n, start=0)
    Solution().houseCount(st, k)
    print(f"n={n}, k={k}: số di chuyển = {st.moves()} (= k + n = {k + n})")

n=5, k=5: số di chuyển = 10 (= k + n = 10)
n=5, k=50: số di chuyển = 55 (= k + n = 55)
n=50, k=1000: số di chuyển = 1050 (= k + n = 1050)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. thử từng `m` | dọn + đặt dấu + đi `m` bước, lặp `m = 1..k` | `O(n·k)` | `O(1)` | hiểu đề, nhưng chậm |
| 2. đóng hết, dấu mở | dọn đóng `k` nhà, mở 1 nhà, đi tới khi gặp mở | `O(k)` | `O(1)` | **cách nộp bài khuyến nghị** |
| 3. mở hết, dấu đóng | đối xứng cách 2 | `O(k)` | `O(1)` | nhớ rằng dấu có thể là cửa đóng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(3 + 4) % 5`, `(0 - 1) % 4`, `[i % 3 for i in range(5)]`, và `for step in range(1, 4)` in ra những giá trị nào.

**Bài 2 (tính tay).** Đường có `n = 3` nhà, cửa `[1, 1, 0]`, bạn đứng ở nhà 1, `k = 4`. Viết trạng thái sau pha dọn (nhà nào đang đứng?), sau khi đặt dấu, và số bước pha đếm.

**Bài 3 (điền TODO).** Hoàn thành `house_count_todo(street, k)`.

In [29]:
def house_count_todo(street, k):
    # TODO 1: lặp k lần: đóng cửa rồi sang phải
    # TODO 2: mở cửa nhà hiện tại làm dấu
    # TODO 3: lặp step = 1..k: sang phải; nếu cửa mở thì return step
    # TODO 4: return -1 (không xảy ra khi n <= k)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `house_count_open2(street, k)` bằng cách **mở hết rồi đặt dấu đóng** mà không nhìn Cách 3. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `house_count_and_moves(street, k)` trả về cặp `(n, số lần di chuyển)` bằng cách dùng `street.moves()` của lớp mô phỏng; kiểm tra số lần di chuyển đúng bằng `k + n`. (b) **Nâng cao:** viết `house_count_unknown_k(street)` **không dùng `k`** nhưng được dùng `moveLeft()`: đặt dấu mở ở nhà xuất phát; mỗi lần sang phải `i` bước, nếu thấy cửa mở thì **đóng nó rồi quay về trái `i` bước**; nếu cửa nhà xuất phát giờ đã đóng thì trả `i`, ngược lại quay lại phải `i` bước và tiếp tục.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7 % 5 = 2`; `(−1) % 4 = 3`; `[0, 1, 2, 0, 1]`; `step` chạy `1, 2, 3`. Bài 2: pha dọn đóng `k = 4` nhà liên tiếp bắt đầu từ nhà 1: các nhà `1, 2, 0, 1`; kết thúc đứng ở nhà `(1 + 4) % 3 = 2`.
- **Mức 2 (vừa):** Bài 3: ba đoạn: `for _ in range(k): closeDoor(); moveRight()`, `openDoor()`, rồi `for step in range(1, k + 1): moveRight(); if isDoorOpen(): return step`. Bài 4: đổi `closeDoor ↔ openDoor` và `if not isDoorOpen()`.
- **Mức 3 (sâu):** Bài 5(a): gọi `n = Solution().houseCount(street, k)` rồi `return n, street.moves()`. Bài 5(b): vòng ngoài `while True`; sau mỗi lần sang phải, nếu `isDoorOpen()`: `closeDoor()`, `moveLeft()` `i` lần, rồi nếu `not isDoorOpen()` thì trả `i`; còn không thì `moveRight()` `i` lần để quay lại vị trí đang xét.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
assert (3 + 4) % 5 == 2
assert (0 - 1) % 4 == 3
assert [i % 3 for i in range(5)] == [0, 1, 2, 0, 1]
assert list(range(1, 4)) == [1, 2, 3]

# Bài 2: n = 3, doors [1,1,0], start = 1, k = 4
st = Street([1, 1, 0], start=1)
for _ in range(4):
    st.closeDoor(); st.moveRight()
assert st._pos == (1 + 4) % 3 == 2                     # đứng ở nhà 2 (kiểm tra nội bộ cho bài tập)
assert st._doors == [0, 0, 0]
st.openDoor()
assert st._doors == [0, 0, 1]
steps = 0
for step in range(1, 5):
    st.moveRight()
    if st.isDoorOpen():
        steps = step
        break
assert steps == 3

# Bài 3
def house_count_todo_ans(street, k):
    for _ in range(k):
        street.closeDoor()
        street.moveRight()
    street.openDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if street.isDoorOpen():
            return step
    return -1

assert house_count_todo_ans(Street([0, 0, 0, 0]), 10) == 4
assert house_count_todo_ans(Street([1, 0, 1, 1, 0]), 5) == 5

# Bài 4
def house_count_open2(street, k):
    for _ in range(k):
        street.openDoor()
        street.moveRight()
    street.closeDoor()
    for step in range(1, k + 1):
        street.moveRight()
        if not street.isDoorOpen():
            return step
    return -1

assert house_count_open2(Street([0, 0, 0, 0]), 10) == 4
assert house_count_open2(Street([1, 0, 1, 1, 0]), 5) == 5

# Bài 5(a)
def house_count_and_moves(street, k):
    n = Solution().houseCount(street, k)
    return n, street.moves()

for doors, k in [([1, 0, 1, 1, 0], 5), ([0], 9), ([1] * 20, 30)]:
    n, mv = house_count_and_moves(Street(doors, start=0), k)
    assert n == len(doors) and mv == k + n

# Bài 5(b): không dùng k, được dùng moveLeft
def house_count_unknown_k(street):
    street.openDoor()                      # dấu ở nhà xuất phát
    i = 0
    while True:
        street.moveRight()
        i += 1
        if street.isDoorOpen():
            street.closeDoor()
            for _ in range(i):
                street.moveLeft()
            if not street.isDoorOpen():    # nhà xuất phát giờ đóng => vừa đóng chính nó
                return i
            for _ in range(i):
                street.moveRight()

for doors, start in [([0, 0, 0, 0], 0), ([1, 0, 1, 1, 0], 3), ([1], 0), ([1, 1, 1], 1), ([0, 1, 0, 1, 1, 1, 0], 4)]:
    assert house_count_unknown_k(Street(doors, start=start, move_limit=10000)) == len(doors), (doors, start)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (làm sạch → đặt dấu → đếm)

```python
from typing import Optional

class Solution:
    def houseCount(self, street: Optional['Street'], k: int) -> int:
        for _ in range(k):          # dọn: đóng mọi cửa (k >= n)
            street.closeDoor()
            street.moveRight()
        street.openDoor()           # đặt dấu duy nhất
        for step in range(1, k + 1):
            street.moveRight()      # dịch trước...
            if street.isDoorOpen(): # ...rồi hỏi
                return step
        return -1
```

**Checklist**

- [ ] Nhớ rằng cửa ban đầu có thể mở/đóng tuỳ ý (không giả sử)?
- [ ] Dọn **đủ `k` bước** (vì chỉ biết `n ≤ k`)?
- [ ] Đặt **đúng một dấu** (mở nếu dọn bằng đóng, hoặc ngược lại)?
- [ ] **Dịch sang phải rồi mới hỏi** `isDoorOpen()`?
- [ ] Trả về số bước ở **pha đếm** (không phải `k`)?
- [ ] Dùng vòng lặp **có cận** để tránh treo?
- [ ] Đã thử `n = 1`, `k = n`, `k` rất lớn, xuất phát ở nhà bất kỳ?
- [ ] Đã đối chiếu đề / ví dụ / danh sách hàm của `Street` với đề gốc (bài Premium)?